In [10]:
from IPython.display import display, Image, HTML
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/My Drive/Crime Analysis/')
%ls

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
bunseki_conpe/  data/  final_data/  tochijihai_hackathon/


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd

# DataFrame Initializations

Common DataFrames of target labels and subregion polygons (with some basic features)

In [3]:
target_df = pd.read_csv('data/region/region_2024_total.csv').set_index('市区町丁', drop=True, inplace=False)
target_df = target_df.drop(['latitude', 'longitude'], axis=1)

def convert_kanji_to_num(s):
    if pd.isna(s):
        return s
    return s.replace('〇', '０').replace('一', '１').replace('二', '２').replace('三', '３').replace('四', '４').replace('五', '５').replace('六', '６').replace('七', '７').replace('八', '８').replace('九', '９')

subregions = gpd.read_file('data/subregions.geojson')
subregions = subregions.to_crs(epsg=4326)
subregions['市区町丁'] = subregions.apply(lambda row: f"{row['CITY_NAME']}{convert_kanji_to_num(row['S_NAME'])}", axis=1)
subregions = subregions.set_index('市区町丁', drop=True, inplace=False)
subregions = subregions.loc[~subregions.index.duplicated(keep='first')]

common_regions = list(set(target_df.index) & set(subregions.index))
common_regions = pd.Index(common_regions).unique()
target_df = target_df.loc[common_regions]
subregions = subregions.loc[common_regions]

subregions = subregions.to_crs('EPSG:3099')

In [4]:
def read_gpd(file_path, **kwargs):
    try:
        gdf = gpd.read_file(file_path, **kwargs)
        return gdf
    except Exception as e:
        print(f"Error reading the GeoJSON file: {e}")
        return None

Use the function below for any GDF with shapely points

In [5]:
def calculate_areas_in_subregions(regions, subregions, categorical_columns):
    """
    Calculate the proportion of area in each subregion covered by different categories.

    Parameters:
    - regions (GeoDataFrame): A GeoDataFrame containing polygon geometries of regions with categories.
    - subregions (GeoDataFrame): A GeoDataFrame containing polygon geometries of subregions.
    - categorical_columns (list): A list of column names in 'regions' to be used for area calculation.

    Returns:
    - pd.DataFrame: A DataFrame with subregions as the index and area proportions of each category as columns.
    """
    if regions.crs == None:
        regions.set_crs(epsg=4326, inplace=True)
    if regions.crs != subregions.crs:
        regions = regions.to_crs(subregions.crs)

    regions_in_subregions = gpd.sjoin(regions, subregions, how="inner", predicate="intersects")
    regions_in_subregions['area'] = regions_in_subregions.geometry.area
    result = pd.DataFrame(index=subregions.index)

    for column in categorical_columns:
        aggregated = regions_in_subregions.groupby(['index_right', column])['area'].sum().unstack(fill_value=0)
        total_area_by_subregion = subregions.geometry.area
        proportions = aggregated.div(total_area_by_subregion, axis=0)
        result = result.join(proportions, how='left').fillna(0)

    return result

In [21]:
def calculate_area_featuresum_in_subregions(regions, subregions, numerical_columns):
    """
    Calculate the sum of numerical values within each subregion based on the regions' data.

    Parameters:
    - regions (GeoDataFrame): A GeoDataFrame containing polygon geometries of regions.
    - subregions (GeoDataFrame): A GeoDataFrame containing polygon geometries of subregions.
    - numerical_columns (list): A list of numerical column names in 'regions' to be summed.

    Returns:
    - pd.DataFrame: A DataFrame with subregions as the index and sums of numerical values as columns.
    """
    if regions.crs == None:
        regions.set_crs(epsg=4326, inplace=True)
    if regions.crs != subregions.crs:
        regions = regions.to_crs(subregions.crs)

    regions_in_subregions = gpd.sjoin(regions, subregions, how="inner", predicate="intersects")
    regions_in_subregions['region_area'] = regions_in_subregions.geometry.area
    subregions['subregion_area'] = subregions.geometry.area
    regions_in_subregions = regions_in_subregions.merge(subregions[['subregion_area']], left_on='index_right', right_index=True)
    display(regions_in_subregions)
    regions_in_subregions['area_proportion'] = regions_in_subregions['region_area'] / regions_in_subregions['subregion_area']

    result = pd.DataFrame(index=subregions.index)

    for column in numerical_columns:
        aggregated = regions_in_subregions.groupby('index_right')[column].sum()
        result[column] = aggregated

    result['proportion'] = regions_in_subregions.groupby('index_right')['area_proportion'].sum()

    result = result.fillna(0)

    return result

In [25]:
def calculate_area_coverage_proportion_in_subregions(regions, subregions):
    """
    Calculate the proportion of each subregion's area that is covered by any of the regions.

    Parameters:
    - regions (GeoDataFrame): A GeoDataFrame containing polygon geometries of regions.
    - subregions (GeoDataFrame): A GeoDataFrame containing polygon geometries of subregions.

    Returns:
    - pd.DataFrame: A DataFrame with subregions as the index and the coverage proportion (0-1) as the column.
    """
    if regions.crs is None:
        regions.set_crs(epsg=4326, inplace=True)
    if regions.crs != subregions.crs:
        regions = regions.to_crs(subregions.crs)

     # Perform a spatial join to find which regions intersect with which subregions
    regions_in_subregions = gpd.sjoin(regions, subregions, how='inner', predicate='intersects')

    # Rename the geometry column of the subregions to geometry_right
    regions_in_subregions = regions_in_subregions.rename(columns={'geometry': 'geometry_left', 'geometry_right': 'geometry_right'})

    # Calculate the area of the intersection between each region and subregion
    regions_in_subregions['intersection_area'] = regions_in_subregions.geometry_left.intersection(regions_in_subregions.geometry_right).area

    # Calculate the total area of each subregion
    subregion_areas = regions_in_subregions.groupby('KEY_CODE')['geometry_right'].first().area

    # Calculate the proportion of each region covered by each subregion
    regions_in_subregions['coverage_proportion'] = regions_in_subregions['intersection_area'] / subregion_areas

    # Return the result
    return regions_in_subregions[['KEY_CODE', 'coverage_proportion']]

### Forest

In [ ]:
gdfs = []
for i in range(1, 5):
    forest_gdf_temp = read_gpd(f'bunseki_conpe/datasets/area_data/forests_{i}.shp')
    forest_gdf_temp['属性'] = i
    gdfs.append(forest_gdf_temp)

forests = pd.concat(gdfs, ignore_index=True).set_crs(epsg=4326, inplace=True).to_crs('EPSG:3099')

In [ ]:
forests_aggregate = calculate_areas_in_subregions(forests, subregions, ['属性'])

### DID Population Concentration Areas

In [17]:
populationconcentration = read_gpd('bunseki_conpe/datasets/area_data/populationconcentration.shp', encoding='utf-8')
rename_cols = {
    'A16_005': '人口',
    'A16_006': '面積',
    'A16_009': '人口割合',
    'A16_010': '面積割合',
    'A16_012': '男',
    'A16_013': '女',
    'A16_014': '世帯数'
}
populationconcentration = populationconcentration.rename(columns=rename_cols)

In [22]:
populationconcentration_aggregate = calculate_area_featuresum_in_subregions(populationconcentration, subregions, list(rename_cols.values()))

,A16_001,A16_002,A16_003,A16_004,人口,面積,A16_007,A16_008,人口割合,面積割合,...,KIGO_I,KBSUM,JINKO,SETAI,X_CODE,Y_CODE,KCODE1,subregion_area_x,region_area,subregion_area_y
0,1310901,13109,品川区,1,422488,22.84,386855,22.84,100.0,100.0,...,None,60,4911,3275,139.728863,35.586732,0070-01,173016.233954,2.267841e+07,173016.233954
13,1311101,13111,大田区,1,748081,61.86,717082,60.66,100.0,100.0,...,None,60,4911,3275,139.728863,35.586732,0070-01,173016.233954,6.182512e+07,173016.233954
0,1310901,13109,品川区,1,422488,22.84,386855,22.84,100.0,100.0,...,None,79,6755,4843,139.734497,35.589852,0230-03,159957.864409,2.267841e+07,159957.864409
13,1311101,13111,大田区,1,748081,61.86,717082,60.66,100.0,100.0,...,None,79,6755,4843,139.734497,35.589852,0230-03,159957.864409,6.182512e+07,159957.864409
0,1310901,13109,品川区,1,422488,22.84,386855,22.84,100.0,100.0,...,None,60,5252,3407,139.731481,35.590337,0230-06,197871.878475,2.267841e+07,197871.878475
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53,1322503,13225,稲城市,3,11264,1.05,-99999,-99999.00,12.1,5.8,...,None,23,2987,889,139.467921,35.623153,0100-03,259970.916148,1.035123e+06,259970.916148
54,1322502,13225,稲城市,2,11760,1.16,10586,0.99,12.6,6.5,...,None,91,3298,1279,139.494703,35.607304,0060-02,395573.397508,1.176549e+06,395573.397508
54,1322502,13225,稲城市,2,11760,1.16,10586,0.99,12.6,6.5,...,None,54,4216,2463,139.488064,35.609599,0060-03,347691.601648,1.176549e+06,347691.601648
54,1322502,13225,稲城市,2,11760,1.16,10586,0.99,12.6,6.5,...,None,60,2750,1258,139.492207,35.612890,0060-01,398703.514223,1.176549e+06,398703.514223


KeyError: 'subregion_area'

In [26]:
populationconcentration_aggregate = calculate_area_coverage_proportion_in_subregions(populationconcentration, subregions)

AttributeError: 'GeoDataFrame' object has no attribute 'geometry_right'

In [9]:
populationconcentration_aggregate

,人口,面積,人口割合,面積割合,男,女,世帯数,proportion
府中市住吉町４丁目,262790.0,29.43,100.0,100.0,131468.0,131322.0,123931.0,194.321224
東大和市蔵敷２丁目,83156.0,9.70,99.1,72.3,40664.0,42492.0,36050.0,43.604454
世田谷区深沢８丁目,943664.0,58.05,100.0,100.0,445592.0,498072.0,492065.0,435.735084
品川区東中延２丁目,422488.0,22.84,100.0,100.0,208688.0,213800.0,237641.0,373.904331
北区堀船１丁目,355213.0,20.61,100.0,100.0,176289.0,178924.0,189700.0,146.795656
...,...,...,...,...,...,...,...,...
新宿区神楽坂５丁目,349385.0,18.22,100.0,100.0,174822.0,174563.0,222800.0,925.756030
葛飾区青戸２丁目,453093.0,34.80,100.0,100.0,225758.0,227335.0,215948.0,209.544892
品川区南品川２丁目,422488.0,22.84,100.0,100.0,208688.0,213800.0,237641.0,162.295941
足立区大谷田１丁目,695043.0,53.25,100.0,100.0,347408.0,347635.0,345346.0,213.109823


In [ ]:
calculate_area_featuresum_in_subregions(populationconcentration, subregions, list(rename_cols.values()))

<ipython-input-61-5260f48cef57>:3: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: EPSG:4326
Right CRS: EPSG:3099

  intersections = gpd.overlay(regions, subregions, how='intersection')
<ipython-input-61-5260f48cef57>:6: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  intersections['area'] = intersections.geometry.area


KeyError: 'index_right'

## Correlation / Relation metrics

In [ ]:
# Pearson Correlation
combined_df = policestations_aggregate.join(target_df)
correlation_matrix = combined_df.corr(method='pearson')

target_correlations = correlation_matrix[target_df.columns]
target_correlations.to_csv('bunseki_conpe/correlation_matrix.csv', index=True)

In [ ]:
# Mutual Information Regression
from sklearn.feature_selection import mutual_info_regression

mi_scores_df = pd.DataFrame()

for target in target_df.columns:
    y = target_df[target]

    mi = mutual_info_regression(facilities_aggregate, y)

    mi_scores = pd.DataFrame({'Feature': facilities_aggregate.columns, 'Mutual Information': mi})
    mi_scores['Target'] = target
    mi_scores_df = pd.concat([mi_scores_df, mi_scores], axis=0)

mi_scores_df = mi_scores_df.pivot(index='Target', columns='Feature', values='Mutual Information')
mi_scores_df.to_csv('bunseki_conpe/mi_scores.csv', index=True)

In [ ]:
n_shuffles = 1000  # Takes about 4:30
shuffled_mi_scores = []

for i in range(n_shuffles):
    if i % 100 == 0:
        print(f"Shuffling iteration {i}/{n_shuffles}")
    shuffled_y = target_df[target].sample(frac=1, random_state=i).reset_index(drop=True)  # Shuffle target
    mi_shuffled = mutual_info_regression(facilities_aggregate, shuffled_y)
    shuffled_mi_scores.extend(mi_shuffled)

shuffled_mi_scores = pd.DataFrame(shuffled_mi_scores, columns=['Mutual Information'])
threshold = shuffled_mi_scores['Mutual Information'].quantile(0.95)

print(f"Threshold MI score based on shuffled data (95th percentile): {threshold}")

mi_scores_df = mi_scores_df.melt(var_name='Feature', value_name='Mutual Information', ignore_index=False).reset_index()
significant_mi_scores = mi_scores_df[mi_scores_df['Mutual Information'] > threshold]

print("Significant MI scores based on threshold:")
significant_mi_scores

Shuffling iteration 0/1000
Shuffling iteration 100/1000
Shuffling iteration 200/1000
Shuffling iteration 300/1000
Shuffling iteration 400/1000
Shuffling iteration 500/1000
Shuffling iteration 600/1000
Shuffling iteration 700/1000
Shuffling iteration 800/1000
Shuffling iteration 900/1000
Threshold MI score based on shuffled data (95th percentile): 0.018774915548637805
Significant MI scores based on threshold:


,Target,Feature,Mutual Information
54,凶悪犯計,公民館の種別_中央,0.019864
70,非侵入窃盗自販機ねらい,公民館の種別_中央,0.020105
102,非侵入窃盗オートバイ盗,公民館の種別_分館,0.019133
166,粗暴犯傷害,施設区分コード_その他集客施設,0.020872
179,非侵入窃盗置引き,施設区分コード_その他集客施設,0.020702
188,その他計,施設区分コード_公会堂・集会場,0.022897
210,非侵入窃盗すり,施設区分コード_公会堂・集会場,0.024725
211,非侵入窃盗その他,施設区分コード_公会堂・集会場,0.019741
219,非侵入窃盗自転車盗,施設区分コード_公会堂・集会場,0.025081
225,その他計,施設区分コード_劇場・演劇場,0.029078
